In [ ]:
# 新增欄位，用於Other

import pandas as pd
import numpy as np
import os
import glob

def augment_imu_data(input_dir, output_dir, noise_std=0.5, scale_factor=1.0):
    """
    對資料夾內的 IMU CSV 數據進行資料增強。
    
    參數 (閾值可供調整):
    - input_dir (str): 原始 CSV 檔案所在的資料夾路徑。
    - output_dir (str): 增強後 CSV 檔案的輸出資料夾路徑。
    - noise_std (float): 高斯雜訊的標準差。值越大，隨機晃動/雜訊越明顯 (適合做邊界測試)。
    - scale_factor (float): 振幅縮放係數。1.0為不變，1.2表示動作幅度放大20% (適合做誇張化動作)。
    """
    
    # 確保輸出資料夾存在
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    # 取得所有 CSV 檔案
    csv_files = glob.glob(os.path.join(input_dir, "*.csv"))
    
    if not csv_files:
        print(f"在 {input_dir} 找不到任何 CSV 檔案。")
        return

    # 定義需要增強的感測器基本名稱 (不含 X, Y, Z)
    base_sensors = ['acceleration', 'gyroscope']
    axes = ['_X', '_Y', '_Z']
    
    print(f"找到 {len(csv_files)} 個檔案，準備開始處理...")

    for file_path in csv_files:
        file_name = os.path.basename(file_path)
        output_path = os.path.join(output_dir, f"aug_{file_name}")
        
        # 讀取原始資料
        try:
            df = pd.read_csv(file_path)
        except Exception as e:
            print(f"讀取 {file_name} 失敗: {e}")
            continue
            
        # 產生所有目標欄位名稱 (包含未矯正與已矯正)
        cols_to_augment = []
        for sensor in base_sensors:
            for axis in axes:
                cols_to_augment.append(f"{sensor}{axis}")             # 未矯正
                cols_to_augment.append(f"{sensor}{axis}_corrected")   # 已矯正
                
        # 針對選定欄位進行增強
        for col in cols_to_augment:
            if col in df.columns:
                # 新增強的欄位名稱
                aug_col_name = f"{col}_aug"
                
                # 方法 1: 加上高斯雜訊 (模擬不穩定或隨機的晃動)
                noise = np.random.normal(loc=0.0, scale=noise_std, size=len(df))
                
                # 方法 2: 振幅縮放 (模擬動作幅度變大或變小)
                # 將原數值乘上 scale_factor 後加上 noise
                df[aug_col_name] = (df[col] * scale_factor) + noise
                
        # 將結果存入新檔案
        df.to_csv(output_path, index=False)
        print(f"已處理並儲存: {output_path} (新增了 {len([c for c in df.columns if '_aug' in c])} 個增強欄位)")

    print("\n所有檔案處理完成！")

# ==========================================
# 參數設定區 (請依照你的實驗需求調整這裡的數值)
# ==========================================
if __name__ == "__main__":
    
    # 1. 設定資料夾路徑 (可以使用相對路徑或絕對路徑)
    INPUT_FOLDER = "./data"      # 放置你原始 CSV 的資料夾
    OUTPUT_FOLDER = "./aug_data"     # 處理後輸出的資料夾

    # 2. 設定增強閾值 (核心控制項)
    # 微小增強 (訓練集擴充)：NOISE_STD = 0.2, SCALE_FACTOR = 1.05
    # 誇張增強 (製造第三類別)：NOISE_STD = 1.0, SCALE_FACTOR = 1.5
    NOISE_STD = 2.0         # 雜訊強度 (標準差)
    SCALE_FACTOR = 1.2      # 動作放大倍率 (1.0 代表不放大，1.2代表放大20%)

    # 執行增強程式
    augment_imu_data(
        input_dir=INPUT_FOLDER, 
        output_dir=OUTPUT_FOLDER, 
        noise_std=NOISE_STD, 
        scale_factor=SCALE_FACTOR
    )

找到 91 個檔案，準備開始處理...
已處理並儲存: ./aug_data/aug_1772435987067_nn_160hz_large_esp32_t71_v10_Myasthenia_pre2_Tired.csv (新增了 12 個增強欄位)
已處理並儲存: ./aug_data/aug_1772434858572_zhao_160hz_all_esp32_t83_v10_notTired_pre.csv (新增了 12 個增強欄位)
已處理並儲存: ./aug_data/aug_1772435382146_nn_160hz_small_esp32_t494_v10_notTired_pre1.csv (新增了 12 個增強欄位)
已處理並儲存: ./aug_data/aug_1772435382146_nn_160hz_small_esp32_t494_v10_notTired_pre2.csv (新增了 12 個增強欄位)
已處理並儲存: ./aug_data/aug_1772435241962_zhao_160hz_small_esp32_t371_v10_notTired_pre.csv (新增了 12 個增強欄位)
已處理並儲存: ./aug_data/aug_1772434858490_nn_160hz_all_esp32_t89_v10_notTired_pre.csv (新增了 12 個增強欄位)
已處理並儲存: ./aug_data/aug_1772435425028_zhao_160hz_small_esp32_t156_v10_notTired_pre.csv (新增了 12 個增強欄位)
已處理並儲存: ./aug_data/aug_1772435956364_zhao_160hz_large_esp32_t64_v10_notTired_pre.csv (新增了 12 個增強欄位)
已處理並儲存: ./aug_data/aug_1772436106977_zhao_160hz_stairs_esp32_t126_v10_Tachypnea_pre_Tired.csv (新增了 12 個增強欄位)
已處理並儲存: ./aug_data/aug_1772435987067_nn_160hz_large_esp32_t71_v10_no

In [4]:
# 直接取代數據，用於增強原數據

import pandas as pd
import numpy as np
import os
import glob

def augment_imu_data(input_dir, output_dir, noise_std=0.5, scale_factor=1.0):
    """
    對資料夾內的 IMU CSV 數據進行資料增強 (直接取代舊欄位)。
    
    參數 (閾值可供調整):
    - input_dir (str): 原始 CSV 檔案所在的資料夾路徑。
    - output_dir (str): 增強後 CSV 檔案的輸出資料夾路徑。
    - noise_std (float): 高斯雜訊的標準差。值越大，隨機晃動/雜訊越明顯 (適合做邊界測試)。
    - scale_factor (float): 振幅縮放係數。1.0為不變，1.2表示動作幅度放大20% (適合做誇張化動作)。
    """
    
    # 確保輸出資料夾存在
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    # 取得所有 CSV 檔案
    csv_files = glob.glob(os.path.join(input_dir, "*.csv"))
    
    if not csv_files:
        print(f"在 {input_dir} 找不到任何 CSV 檔案。")
        return

    # 定義需要增強的感測器基本名稱 (不含 X, Y, Z)
    base_sensors = ['acceleration', 'gyroscope']
    axes = ['_X', '_Y', '_Z']
    
    print(f"找到 {len(csv_files)} 個檔案，準備開始處理...")

    for file_path in csv_files:
        file_name = os.path.basename(file_path)
        output_path = os.path.join(output_dir, f"aug2_{file_name}")
        
        # 讀取原始資料
        try:
            df = pd.read_csv(file_path)
        except Exception as e:
            print(f"讀取 {file_name} 失敗: {e}")
            continue
            
        # 產生所有目標欄位名稱 (包含未矯正與已矯正)
        cols_to_augment = []
        for sensor in base_sensors:
            for axis in axes:
                cols_to_augment.append(f"{sensor}{axis}")             # 未矯正
                cols_to_augment.append(f"{sensor}{axis}_corrected")   # 已矯正
                
        # 紀錄實際修改的欄位數量
        modified_count = 0
        
        # 針對選定欄位進行增強
        for col in cols_to_augment:
            if col in df.columns:
                # 方法 1: 加上高斯雜訊 (模擬不穩定或隨機的晃動)
                noise = np.random.normal(loc=0.0, scale=noise_std, size=len(df))
                
                # 方法 2: 振幅縮放 (模擬動作幅度變大或變小)
                # 將原數值乘上 scale_factor 後加上 noise，並直接覆寫回原本的欄位
                df[col] = (df[col] * scale_factor) + noise
                
                modified_count += 1
                
        # 將結果存入新檔案
        df.to_csv(output_path, index=False)
        print(f"已處理並儲存: {output_path} (修改了 {modified_count} 個原始欄位)")

    print("\n所有檔案處理完成！")

# ==========================================
# 參數設定區 (請依照你的實驗需求調整這裡的數值)
# ==========================================
if __name__ == "__main__":
    
    # 1. 設定資料夾路徑 (可以使用相對路徑或絕對路徑)
    INPUT_FOLDER = "./data"      # 放置你原始 CSV 的資料夾
    OUTPUT_FOLDER = "./aug_data" # 處理後輸出的資料夾

    # 2. 設定增強閾值 (核心控制項)
    # 微小增強 (訓練集擴充)：NOISE_STD = 0.2, SCALE_FACTOR = 1.05
    # 誇張增強 (製造第三類別)：NOISE_STD = 1.0, SCALE_FACTOR = 1.5
    NOISE_STD = 0.3         # 雜訊強度 (標準差)
    SCALE_FACTOR = 1.1      # 動作放大倍率 (1.0 代表不放大，1.2代表放大20%)

    # 執行增強程式
    augment_imu_data(
        input_dir=INPUT_FOLDER, 
        output_dir=OUTPUT_FOLDER, 
        noise_std=NOISE_STD, 
        scale_factor=SCALE_FACTOR
    )

找到 91 個檔案，準備開始處理...
已處理並儲存: ./aug_data/aug2_1772780313498_zhao_160hz_stairs_esp32_t198_v10_Tachypnea_pre_Tired.csv (修改了 12 個原始欄位)
已處理並儲存: ./aug_data/aug2_1772781192510_zhao_160hz_large_esp32_t106_v10_notTired_pre.csv (修改了 12 個原始欄位)
已處理並儲存: ./aug_data/aug2_1772442452640_zhao_160hz_stairs_esp32_t136_v10_notTired_pre.csv (修改了 12 個原始欄位)
已處理並儲存: ./aug_data/aug2_1772441079471_nn_160hz_small_esp32_t36_v10_notTired_pre.csv (修改了 12 個原始欄位)
已處理並儲存: ./aug_data/aug2_1772783441136_zhao_160hz_stairs_esp32_t111_v10_notTired_pre.csv (修改了 12 個原始欄位)
已處理並儲存: ./aug_data/aug2_1772865119196_nn_160hz_all_esp32_t48_v10_notTired_pre.csv (修改了 12 個原始欄位)
已處理並儲存: ./aug_data/aug2_1772438151203_zhao_160hz_stairs_esp32_t178_v10_Tachypnea_pre_Tired.csv (修改了 12 個原始欄位)
已處理並儲存: ./aug_data/aug2_1772439585508_zhao_160hz_stairs_esp32_t94_v10_Tachypnea_pre_Tired.csv (修改了 12 個原始欄位)
已處理並儲存: ./aug_data/aug2_1772436195564_nn_160hz_stairs_esp32_t158_v10_Myasthenia_pre1_Tired.csv (修改了 12 個原始欄位)
已處理並儲存: ./aug_data/aug2_1772783442055